In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [7]:
df = pd.read_csv("customer_churn_dataset-testing-master.csv")

In [8]:
df = df.dropna()
cat_cols = ['Gender', 'Subscription Type', 'Contract Length']
for col in cat_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

In [9]:
X = df.drop(columns=['Churn', 'CustomerID'] if 'CustomerID' in df.columns else ['Churn'])
y = df['Churn']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
baseline_model = LogisticRegression(random_state=42)
baseline_model.fit(X_train_scaled, y_train)

LogisticRegression(random_state=42)

In [13]:
y_pred_base = baseline_model.predict(X_test_scaled)
y_prob_base = baseline_model.predict_proba(X_test_scaled)[:, 1]

In [14]:
print("--- Baseline Model (Logistic Regression) Performance ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_base):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_base):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_base):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_base):.4f}")

--- Baseline Model (Logistic Regression) Performance ---
Accuracy:  0.8257
Precision: 0.8123
Recall:    0.8219
F1-Score:  0.8171
ROC-AUC:   0.9020


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [16]:
rf = RandomForestClassifier(random_state=42)

In [17]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

In [18]:
random_search = estimator = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=4,
    cv=3,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)
print("Training optimized Random Forest (this may take a moment)...")
random_search.fit(X_train_scaled, y_train)

Training optimized Random Forest (this may take a moment)...


RandomizedSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42),
                   n_iter=4, n_jobs=-1,
                   param_distributions={'max_depth': [10, 20, None],
                                        'min_samples_split': [2, 5],
                                        'n_estimators': [50, 100]},
                   random_state=42, scoring='f1')

In [19]:
best_rf = random_search.best_estimator_

In [20]:
y_pred_opt = best_rf.predict(X_test_scaled)
y_prob_opt = best_rf.predict_proba(X_test_scaled)[:, 1]
print("\n--- Optimized Model (Random Forest) Performance ---")
print(f"Best Parameters: {random_search.best_params_}")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_opt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_opt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_opt):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_opt):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_opt):.4f}")


--- Optimized Model (Random Forest) Performance ---
Best Parameters: {'n_estimators': 100, 'min_samples_split': 2, 'max_depth': None}
Accuracy:  0.9990
Precision: 0.9993
Recall:    0.9985
F1-Score:  0.9989
ROC-AUC:   1.0000


In [21]:
import pickle

In [22]:
with open("model.pkl", "wb") as f:
  pickle.dump(best_rf, f)
print("Model saved successfully as model.pkl!")

Model saved successfully as model.pkl!


In [23]:
from google.colab import files
files.download("model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
print(X.columns.tolist())

['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction']
